# Emotion Classifier — Kaggle Training Notebook (Tasks 4 & 5)

Fine-tunes `distilbert-base-uncased` on the `dair-ai/emotion` dataset and tracks
**two** experiment versions on Weights & Biases, then pushes the best model to
the Hugging Face Hub.

**Before you Run All:**
1. Settings -> Accelerator -> **GPU T4 x2**.
2. Add-ons -> Secrets -> add `WANDB_API_KEY` and `HF_TOKEN` (do **not** hardcode).
3. Edit `HF_MODEL_REPO` in the config cell to your own Hugging Face repo id.


In [ ]:
# 1) Pin the libraries we rely on (Kaggle already ships torch).
!pip install -q -U transformers==4.40.2 datasets==2.19.0 wandb==0.17.0 \
    accelerate==0.30.1 scikit-learn==1.4.2

In [ ]:
# 2) Load secrets from Kaggle Secrets — never hardcode tokens.
import os, re, json, collections
import numpy as np
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)
wandb.login()
print("Secrets loaded; logged in to W&B and Hugging Face.")

In [ ]:
# 3) Project configuration.
BASE_MODEL     = "distilbert-base-uncased"
DATASET        = "dair-ai/emotion"
DATASET_CONFIG = "split"            # 16k train / 2k validation / 2k test
WANDB_PROJECT  = "mlops-assignment3"
MAX_LENGTH     = 128

# >>> CHANGE THIS to your own public HF repo id before running Task 5 <<<
HF_MODEL_REPO  = "your-username/distilbert-emotion-mlops-a3"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

## Data preparation (mirrors `src/data_prep.py`)
Lowercase, strip URLs / @mentions, collapse whitespace, drop empty rows. The
label map is read straight from the dataset's own schema.

In [ ]:
# 4) Load, inspect and clean the dataset.
from datasets import load_dataset

raw = load_dataset(DATASET, DATASET_CONFIG)
print(raw)

class_label = raw["train"].features["label"]
id2label = {i: n for i, n in enumerate(class_label.names)}
label2id = {n: i for i, n in id2label.items()}
print("Labels:", id2label)

URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
WS_RE      = re.compile(r"\s+")

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    return WS_RE.sub(" ", t).strip()

raw = raw.map(lambda b: {"text": clean_text(b["text"])})
raw = raw.filter(lambda b: len(b["text"]) > 0)

for split in raw:
    c = collections.Counter(raw[split]["label"])
    total = sum(c.values())
    print(split, {id2label[k]: f"{c[k]} ({100*c[k]/total:.1f}%)" for k in sorted(c)})

with open("id2label.json", "w") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)

In [ ]:
# 5) Tokenize.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True,
                     padding="max_length", max_length=MAX_LENGTH)

tokenized = raw.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_ds = tokenized["train"]
eval_ds  = tokenized["validation"]
test_ds  = tokenized["test"]
print(train_ds, eval_ds, test_ds, sep="\n")

In [ ]:
# 6) Metrics logged for every evaluation step.
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

In [ ]:
# 7) One reusable training routine, parameterised by a config dict.
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

def train_version(run_name, config):
    run = wandb.init(project=WANDB_PROJECT, name=run_name,
                     config=config, reinit=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=len(id2label),
        id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir                  = f"./results/{run_name}",
        num_train_epochs            = config["epochs"],
        per_device_train_batch_size = config["batch_size"],
        per_device_eval_batch_size  = config["batch_size"],
        learning_rate               = config["learning_rate"],
        weight_decay                = config.get("weight_decay", 0.0),
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        logging_strategy            = "steps",
        logging_steps               = 50,
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        report_to                   = "wandb",
        run_name                    = run_name,
        fp16                        = True,
        seed                        = config.get("seed", 42),
    )

    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=eval_ds,
                      compute_metrics=compute_metrics)
    trainer.train()

    # Final held-out test metrics, also logged to W&B.
    test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
    print(run_name, "test metrics:", test_metrics)
    wandb.log(test_metrics)

    run_id = run.id
    wandb.finish()
    return trainer, test_metrics, run_id

## Version 1 — baseline
`lr=3e-5`, `batch=16`, `epochs=3`, `weight_decay=0`.

In [ ]:
# 8) Run version 1.
config_v1 = {
    "model": BASE_MODEL, "epochs": 3, "batch_size": 16,
    "learning_rate": 3e-5, "weight_decay": 0.0,
    "version": "v1", "platform": "Kaggle", "seed": 42,
}
trainer_v1, metrics_v1, run_id_v1 = train_version("run-v1", config_v1)

## Version 2 — changed hyperparameters
Higher learning rate, larger batch, more epochs, and a little weight decay:
`lr=5e-5`, `batch=32`, `epochs=4`, `weight_decay=0.01`. Changing several knobs at
once is fine for the assignment, but note in the report which change you believe
drove the difference.

In [ ]:
# 9) Run version 2.
config_v2 = {
    "model": BASE_MODEL, "epochs": 4, "batch_size": 32,
    "learning_rate": 5e-5, "weight_decay": 0.01,
    "version": "v2", "platform": "Kaggle", "seed": 42,
}
trainer_v2, metrics_v2, run_id_v2 = train_version("run-v2", config_v2)

## Task 5 — push the best model & log its URL to W&B

In [ ]:
# 10) Pick the better run by test F1, push it, and record the HF URL.
if metrics_v1["test_f1"] >= metrics_v2["test_f1"]:
    best_name, best_trainer, best_run_id = "run-v1", trainer_v1, run_id_v1
else:
    best_name, best_trainer, best_run_id = "run-v2", trainer_v2, run_id_v2
print("Best version:", best_name)

best_trainer.model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

model_url = f"https://huggingface.co/{HF_MODEL_REPO}"
print("Pushed:", model_url)

# Re-open the best run and write the HF URL into its summary.
run = wandb.init(project=WANDB_PROJECT, id=best_run_id, resume="must")
run.summary["huggingface_model"] = model_url
run.summary["best_version"]      = best_name
wandb.finish()
print("Logged HF model URL to W&B run summary.")

## Compare the two runs side by side

In [ ]:
# 11) Quick local comparison table (W&B will also show this).
import pandas as pd
pd.DataFrame({
    "run-v1": metrics_v1,
    "run-v2": metrics_v2,
}).T[["test_accuracy", "test_f1", "test_loss"]]